In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


print("1. Loading Data...")
df = pd.read_csv('IoTpond1.csv')

df = df.dropna()

df_clean = df[['Temperature (C)', 'PH', 'Ammonia(g/ml)', 'Dissolved Oxygen(g/ml)']].copy()
df_clean.columns = ['Temp', 'pH', 'Ammonia', 'DO']


print(f"Rows before cleaning: {len(df_clean)}")
df_clean = df_clean[df_clean['Ammonia'] < 100]
print(f"Rows after removing sensor errors: {len(df_clean)}")


df_clean['Aerator_Status'] = np.where(
    (df_clean['DO'] < 4.0) |
    (df_clean['Ammonia'] > 0.02) |
    (df_clean['pH'] < 6.5) | (df_clean['pH'] > 8.5),
    1,
    0
)

print("\nClass Distribution (0=Safe, 1=Danger):")
print(df_clean['Aerator_Status'].value_counts())


X = df_clean[['Temp', 'pH', 'Ammonia', 'DO']]
y = df_clean['Aerator_Status']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n" + "="*40)
print("🚨 هام جداً: احفظ الأرقام دي للموبايل 🚨")
print(f"MEAN:  {scaler.mean_}")
print(f"SCALE: {scaler.scale_}")
print("="*40 + "\n")


model = keras.Sequential([
    keras.layers.Dense(16, activation='relu', input_shape=(4,)),
    keras.layers.Dense(8, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("Starting Training...")
model.fit(X_train_scaled, y_train, epochs=10, batch_size=32, validation_data=(X_test_scaled, y_test))


converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open('aquasmart_model.tflite', 'wb') as f:
    f.write(tflite_model)

print("\n✅ تم الحفظ! نزل ملف 'aquasmart_model.tflite' دلوقتي.")

1. Loading Data...
Rows before cleaning: 83074
Rows after removing sensor errors: 68847

Class Distribution (0=Safe, 1=Danger):
Aerator_Status
1    68846
0        1
Name: count, dtype: int64

🚨 هام جداً: احفظ الأرقام دي للموبايل 🚨
MEAN:  [24.50439158  7.49248196  5.20236086 12.87806565]
SCALE: [ 0.94112808  0.58237198 13.21520425 13.15985581]



/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Starting Training...
Epoch 1/10
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 1.0000 - loss: 0.1243 - val_accuracy: 1.0000 - val_loss: 9.7055e-05
Epoch 2/10
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 1.0000 - loss: 1.2411e-04 - val_accuracy: 1.0000 - val_loss: 3.0317e-05
Epoch 3/10
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 1.0000 - loss: 2.3374e-04 - val_accuracy: 1.0000 - val_loss: 1.5716e-05
Epoch 4/10
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 1.0000 - loss: 2.9851e-04 - val_accuracy: 1.0000 - val_loss: 1.0443e-05
Epoch 5/10
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 1.0000 - loss: 1.1184e-04 - val_accuracy: 1.0000 - val_loss: 1.0890e-05
Epoch 6/10
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 1.0000 - loss: 5.0577e-04 - val_accuracy: 1.0000 - val_loss: 1.6142e-06
Epoch 7/10
1722/1722 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 1.0000 - loss: 6.2763e-05 - val_accuracy: 1.0000 - val_loss: 4.9836e-06
Epoch 8/10
1722/1